In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
class ColorContext():
    fav_color: str = "green"
    least_fav_color: str = "orange"

In [ ]:
# read from context 
from langchain.tools import ToolRuntime, tool

@tool
def get_fav_color(runtime: ToolRuntime) -> str:
    """get the favourite color of the user."""
    return runtime.context.fav_color

@tool
def get_least_fav_color(runtime: ToolRuntime) -> str:
    """get the least favourite color of the user."""
    return runtime.context.least_fav_color

# context is passed to the agent via the tool calls. 

In [ ]:
# create agent with the tools and pass the context as context schema. 
from langchain.agents import create_agent

agent = create_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[get_fav_color, get_least_fav_color],
    context_schema=ColorContext
)

In [ ]:
# invoke agent and pass context
agent.invoke(
    {"messages": [{"role": "user", "content": "what's my fav color?"}]},
    context=ColorContext(),
)


/Users/mzainkh/Documents/Learning/LangChain/lib/python3.13/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='context', input_value=ColorContext(fav_color='g...east_fav_color='orange'), input_type=ColorContext])
  return self.__pydantic_serializer__.to_python(


{'messages': [HumanMessage(content="what's my fav color?", additional_kwargs={}, response_metadata={}, id='1ce04909-b2df-4306-b7d1-e71d106e0ae6'),
  AIMessage(content=[{'id': 'toolu_01BXZF7BjiVxdS3t5HjBBXgF', 'input': {}, 'name': 'get_fav_color', 'type': 'tool_use', 'caller': {'type': 'direct'}}], additional_kwargs={}, response_metadata={'id': 'msg_01EjmgpPVyHm8L799PoSh3Rv', 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 601, 'output_tokens': 39, 'server_tool_use': None, 'service_tier': 'standard', 'inference_geo': 'not_available'}, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_run--019c5e1d-a373-7033-bed2-83519d75cf45-0', tool_calls=[{'name': 'get_fav_color', 'args': {}, 'id': 'toolu_01BXZF7BjiVxdS3t5HjBBXgF', 'type': 'tool_call'}], inv

In [5]:
# implement state for the agent
from langchain.agents import AgentState

class CustomState(AgentState):
    fav_color: str

In [6]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def update_fav_color(fav_color: str, runtime: ToolRuntime) -> Command:
    """update the favourite color of the user in the state once they have revealed it."""
    return Command[tuple[()]](update={
        "fav_color": fav_color,
        "messages": [ToolMessage("Successfully updated favourite color", tool_call_id=runtime.tool_call_id)]
    })

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="claude-sonnet-4-5-20250929",
    tools=[update_fav_color],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [8]:
agent.invoke(
    {"messages": [{"role": "user", "content": "my favourite color is black."}]},
    {"configurable": {"thread_id": "1"}}
)

{'messages': [HumanMessage(content='my favourite color is black.', additional_kwargs={}, response_metadata={}, id='121ccec5-4ea5-4f8a-9533-5aca25e21909'),
  AIMessage(content=[{'id': 'toolu_012V4A8ghrJiUafWhcHS8nwq', 'input': {'fav_color': 'black'}, 'name': 'update_fav_color', 'type': 'tool_use', 'caller': {'type': 'direct'}}], additional_kwargs={}, response_metadata={'id': 'msg_01TWo9wbJ8nFSwWHEZQyykoo', 'model': 'claude-sonnet-4-5-20250929', 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'input_tokens': 581, 'output_tokens': 59, 'server_tool_use': None, 'service_tier': 'standard', 'inference_geo': 'not_available'}, 'model_name': 'claude-sonnet-4-5-20250929', 'model_provider': 'anthropic'}, id='lc_run--019c6168-22ca-7bb1-b2c1-cbd6f4f44816-0', tool_calls=[{'name': 'update_fav_color', 'args': {'fav_color': 'black'}, 'id': 'toolu